In [1]:
from nnsight.intervention.backends.remote import RemoteBackend
import socketio
from nnsight import LanguageModel


# Pick a model from the list of available models at https://host.edif.ai/status
model_name = "openai-community/gpt2"
model = LanguageModel(model_name)
tokenizer = model.tokenizer


import torch
import os
API_KEY = os.getenv("EDIF_API_KEY", "") # replace EDIF_API_KEY with whatever name you gave the environment variable



# Create and patch RemoteBackend
remote_backend = RemoteBackend(
    model.to_model_key(),
    host="host.edif.ai/edif",
    ssl=True,
    api_key=API_KEY,
    blocking=True,
)

print("model.to_model_key():", model.to_model_key())
# Patch websocket path
original_connect = socketio.SimpleClient.connect
def patched_connect(self, *args, **kwargs):
    kwargs['socketio_path'] = "/edif/ws/socket.io"
    return original_connect(self, *args, **kwargs)

original_blocking_request = remote_backend.blocking_request
def patched_blocking_request(graph):
    socketio.SimpleClient.connect = patched_connect
    try:
        return original_blocking_request(graph)
    finally:
        socketio.SimpleClient.connect = original_connect

remote_backend.blocking_request = patched_blocking_request






d:\MAAI\MI\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


model.to_model_key(): nnsight.modeling.language.LanguageModel:{"repo_id": "openai-community/gpt2", "revision": "main"}


In [5]:



with model.trace("The Eiffel Tower is in the city of", backend=remote_backend) as tracer:
    model.transformer.h[-1].mlp.output[0][:] = 0
    token_ids = model.lm_head.output.argmax(dim=-1).save()

print(token_ids)
    
import nnsight
print(nnsight.__version__)

[2026-05-03 19:22:02] [aae79c79-02a5-482b-aa0b-5d57a29fd807] RECEIVED   : Your job has been received and is waiting to be queued.
[2026-05-03 19:22:02] [aae79c79-02a5-482b-aa0b-5d57a29fd807] QUEUED     : Your job has been recieved by the coordinator and is waiting to be queued.
[2026-05-03 19:22:02] [aae79c79-02a5-482b-aa0b-5d57a29fd807] QUEUED     : Task aae79c79-02a5-482b-aa0b-5d57a29fd807 has been added to the queue. Currently at position 1
[2026-05-03 19:22:02] [aae79c79-02a5-482b-aa0b-5d57a29fd807] QUEUED     : Dispatching task...
[2026-05-03 19:22:02] [aae79c79-02a5-482b-aa0b-5d57a29fd807] RUNNING    : Your job has started running.
[2026-05-03 19:22:03] [aae79c79-02a5-482b-aa0b-5d57a29fd807] COMPLETED  : Your job has been completed.

tensor([[ 262,   12,  417, 8765,   11,  257,  262, 3504,  286, 3576]])
0.5.3


In [6]:
# used this to train a SAE, mostly default settings, I have not yet tried the SAE, and have not yet tried the more state of the art version with BatchTopKSAE, might try it later

import torch
from sae_lens import (
    
LanguageModelSAERunnerConfig,
LanguageModelSAETrainingRunner,
StandardTrainingSAEConfig,
BatchTopKTrainingSAEConfig,
LoggingConfig,
)

# Define total training steps and batch size
total_training_steps = 30_000
batch_size = 4096
total_training_tokens = total_training_steps * batch_size

# Learning rate and L1 warmup schedules
lr_warm_up_steps = 0
lr_decay_steps = total_training_steps // 5  # 20% of training
l1_warm_up_steps = total_training_steps // 20  # 5% of training

device = "cuda" if torch.cuda.is_available() else "cpu"

# sae_config = BatchTopKTrainingSAEConfig(
#     k=100,
#     d_in=768,
#     d_sae=16384,  
#     apply_b_dec_to_input=False,  
#     normalize_activations="expected_average_only_in",
# )
sae_config = StandardTrainingSAEConfig(
    d_in=768,  # GPT-2 small hook_mlp_out width
    d_sae=16 * 1024,  # the width of the SAE. Larger will result in better stats but slower training.
    apply_b_dec_to_input=True,  # We won't apply the decoder weights to the input.
    normalize_activations="expected_average_only_in",
    l1_coefficient=5,  # will control how sparse the feature activations are
    l1_warm_up_steps=l1_warm_up_steps,  # this can help avoid too many dead features initially.
)


cfg = LanguageModelSAERunnerConfig( 
# Data Generating Function (Model + Training Distribution) 
    model_name="gpt2-small",
    hook_name="blocks.0.hook_mlp_out",  
    dataset_path="apollo-research/roneneldan-TinyStories-tokenizer-gpt2", 
    is_dataset_tokenized=True,  
    streaming=True,  
    # SAE Parameters are in the nested 'sae' config
    sae=sae_config,
    #Training Parameters   
    lr=5e-5, 
    lr_warm_up_steps=lr_warm_up_steps,  
    lr_decay_steps=lr_decay_steps,  
    train_batch_size_tokens=batch_size,  
    #Activation Store Parameter  
    context_size=512,  
    n_batches_in_buffer=64,  
    training_tokens=total_training_tokens,   
    store_batch_size_prompts=16,  
    # WANDB  
    logger=LoggingConfig(      
        log_to_wandb=False,      
        wandb_project="sae_lens_tutorial",      
        wandb_log_frequency=30,     
        eval_every_n_wandb_logs=20, 
    ),  
    # Misc
    device=device,
    seed=42,  
    n_checkpoints=0,  
    checkpoint_path="checkpoints", 
    dtype="float32"
 )
# look at the next cell to see some instruction for what to do while this is running.
sparse_autoencoder = LanguageModelSAETrainingRunner(cfg).run()

sparse_autoencoder.save_inference_model("path/to/save/dir")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 8192.43it/s]
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'apollo-research/roneneldan-TinyStories-tokenizer-gpt2' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loaded pretrained model gpt2-small into HookedTransformer


30000| mse_loss: 31.26758 | l1_loss: 133.08165: 100%|██████████| 122880000/122880000 [33:13<00:00, 61643.92it/s]


(WindowsPath('path/to/save/dir/sae_weights.safetensors'),
 WindowsPath('path/to/save/dir/cfg.json'))